In [1]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [3]:
model = SentenceTransformer("all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
train_df = pd.read_csv("./data/processed/train_clean.csv")
val_df = pd.read_csv("./data/processed/validation_clean.csv")
test_df = pd.read_csv("./data/processed/test_clean.csv")

In [7]:
def create_embeddings(df, batch_size=64):

    texts = df["clean_title"].fillna("").tolist()
    ids = df["id"].tolist()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=True
    )

    embedding_dict = {}

    for image_id, embedding in zip(ids, embeddings):
        embedding_dict[image_id] = embedding.cpu()

    return embedding_dict

In [8]:
train_embeddings = create_embeddings(train_df)
val_embeddings = create_embeddings(val_df)
test_embeddings = create_embeddings(test_df)

Batches:   0%|          | 0/489 [00:00<?, ?it/s]

Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Batches:   0%|          | 0/62 [00:00<?, ?it/s]

In [9]:
import os
import torch

os.makedirs("./data/embeddings", exist_ok=True)

torch.save(train_embeddings, "./data/embeddings/train_embeddings.pt")
torch.save(val_embeddings, "./data/embeddings/validation_embeddings.pt")
torch.save(test_embeddings, "./data/embeddings/test_embeddings.pt")